<cell_type>markdown</cell_type># 高级模型优化技术教程 (Advanced Optimization Tutorial)

> **前置知识**: 量化、剪枝、蒸馏基础，Transformer 架构
>
> **学习目标**: 掌握前沿的模型优化技术，特别是 LLM 优化方法

---

## 为什么需要高级优化技术？

```
┌─────────────────────────────────────────────────────────────┐
│                   高级优化技术的价值                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  基础优化的局限:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  标准 INT8 量化 → LLM 精度损失大 (激活值异常)       │   │
│  │  幅度剪枝 → 可能移除重要权重                        │   │
│  │  标准蒸馏 → 需要额外的教师模型                      │   │
│  │  自回归推理 → 每个 token 都需要完整前向传播         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  高级优化解决方案:                                          │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  混合精度量化 → 敏感层保持高精度                    │   │
│  │  SmoothQuant → 平滑激活值异常，适合 LLM             │   │
│  │  Movement Pruning → 基于训练动态剪枝                │   │
│  │  自蒸馏 → 模型自己作为教师                          │   │
│  │  KV Cache → 缓存避免重复计算                        │   │
│  │  Speculative Decoding → 小模型加速大模型            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **混合精度量化** - 不同层使用不同精度
2. **SmoothQuant** - 平滑激活值异常
3. **Movement Pruning** - 基于训练动态的剪枝
4. **自蒸馏** - 模型自己作为教师
5. **KV Cache 优化** - Transformer 推理加速
6. **Speculative Decoding** - 投机解码加速

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 检测设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print(f"PyTorch 版本: {torch.__version__}")
print(f"使用设备: {device}")

<cell_type>markdown</cell_type>## 1. 混合精度量化

**核心概念**: 不同层对量化的敏感度不同，敏感层保持高精度，不敏感层使用低精度

```
┌─────────────────────────────────────────────────────────────┐
│                   混合精度量化策略                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  层敏感度分析:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  第一层 (输入层): 敏感度高 → FP16                   │   │
│  │  中间层: 敏感度中 → INT8                            │   │
│  │  注意力层: 敏感度高 → FP16                          │   │
│  │  最后一层 (输出层): 敏感度高 → FP16                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  敏感度评估方法:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 逐层量化，测量输出变化 (MSE)                    │   │
│  │  2. 输出变化大 = 敏感层 → 保持高精度                │   │
│  │  3. 输出变化小 = 不敏感层 → 可以量化                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  优点: 在压缩率和精度之间取得更好平衡                       │
│  缺点: 需要额外的敏感度分析步骤                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 混合精度量化实现
# ============================================================

class LayerSensitivityAnalyzer:
    """
    层敏感度分析器
    
    分析每层对量化的敏感度，用于决定混合精度策略
    
    工作原理:
    ┌─────────────────────────────────────────────────────────┐
    │  1. 获取原始模型的基准输出                             │
    │  2. 对每层单独进行量化                                 │
    │  3. 计算量化后输出与基准输出的差异 (MSE)               │
    │  4. MSE 越大，该层越敏感，应保持高精度                 │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, model, calibration_data):
        """
        参数:
            model: 待分析的模型
            calibration_data: 校准数据
        """
        self.model = model
        self.calibration_data = calibration_data
        self.sensitivity = {}
    
    def analyze(self):
        """
        分析每层的量化敏感度
        
        返回:
            dict: {层名称: 敏感度分数}
        """
        self.model.eval()
        
        # 获取基准输出
        with torch.no_grad():
            baseline_output = self.model(self.calibration_data)
        
        # 逐层分析
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                # 保存原始权重
                original_weight = module.weight.data.clone()
                
                # 模拟 INT8 量化
                scale = module.weight.abs().max() / 127
                quantized = torch.round(module.weight / scale) * scale
                module.weight.data = quantized
                
                # 计算量化后的输出
                with torch.no_grad():
                    quant_output = self.model(self.calibration_data)
                
                # 计算输出差异 (MSE)
                mse = F.mse_loss(quant_output, baseline_output).item()
                self.sensitivity[name] = mse
                
                # 恢复原始权重
                module.weight.data = original_weight
        
        return self.sensitivity
    
    def get_precision_config(self, threshold=0.01):
        """
        根据敏感度生成精度配置
        
        参数:
            threshold: 敏感度阈值，超过此值使用 FP16
            
        返回:
            dict: {层名称: 精度}
        """
        config = {}
        for name, sens in self.sensitivity.items():
            if sens > threshold:
                config[name] = 'FP16'  # 敏感层保持高精度
            else:
                config[name] = 'INT8'  # 不敏感层可以量化
        return config


# ============================================================
# 测试模型定义
# ============================================================

class SimpleNet(nn.Module):
    """简单的全连接网络，用于演示"""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


# ============================================================
# 敏感度分析演示
# ============================================================
print("=" * 60)
print("混合精度量化 - 层敏感度分析")
print("=" * 60)

model = SimpleNet()
calibration_data = torch.randn(32, 64)

analyzer = LayerSensitivityAnalyzer(model, calibration_data)
sensitivity = analyzer.analyze()

print("\n层敏感度分析结果:")
print("-" * 40)
for name, sens in sensitivity.items():
    print(f"  {name}: {sens:.6f}")

# 生成精度配置
config = analyzer.get_precision_config(threshold=0.001)
print("\n推荐精度配置:")
print("-" * 40)
for name, precision in config.items():
    print(f"  {name}: {precision}")

<cell_type>markdown</cell_type>## 2. SmoothQuant

**核心概念**: 将激活值的量化难度转移到权重，解决 LLM 中激活值异常值问题

```
┌─────────────────────────────────────────────────────────────┐
│                   SmoothQuant 原理                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  问题: LLM 激活值存在异常值 (outliers)，难以量化            │
│                                                             │
│  激活值分布:                                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  大部分值: [-1, 1]                                  │   │
│  │  异常值: [-100, 100] (少量但影响大)                 │   │
│  │                                                     │   │
│  │  直接量化 → 大部分值被压缩到很小范围 → 精度损失大  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  解决方案: 平滑转移                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Y = XW = (X × diag(s)⁻¹) × (diag(s) × W)          │   │
│  │        = X' × W'                                    │   │
│  │                                                     │   │
│  │  其中 s_j = max(|X_j|)^α / max(|W_j|)^(1-α)        │   │
│  │  α 通常取 0.5                                       │   │
│  │                                                     │   │
│  │  效果: X' 的异常值减小，W' 的范围增大              │   │
│  │        两者都更容易量化                             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  优点: 无需重新训练，适合 LLM                               │
│  缺点: 需要校准数据计算激活值统计                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# SmoothQuant 实现
# ============================================================

def smooth_quant(weight, activation_scale, alpha=0.5):
    """
    SmoothQuant 实现
    
    将激活值的量化难度转移到权重
    
    数学原理:
    ┌─────────────────────────────────────────────────────────┐
    │  Y = XW = (X × diag(s)⁻¹) × (diag(s) × W) = X' × W'   │
    │                                                         │
    │  s_j = max(|X_j|)^α / max(|W_j|)^(1-α)                 │
    │                                                         │
    │  α = 0.5 时，平均分配量化难度                          │
    │  α > 0.5 时，更多难度转移到权重                        │
    │  α < 0.5 时，更多难度保留在激活值                      │
    └─────────────────────────────────────────────────────────┘
    
    参数:
        weight: 权重矩阵 [out_features, in_features]
        activation_scale: 激活值的通道最大值 [in_features]
        alpha: 平滑因子，控制难度分配
        
    返回:
        smoothed_weight: 平滑后的权重
        smooth_factor: 平滑因子 (用于推理时反向缩放激活值)
    """
    # 激活值的通道最大值
    act_max = activation_scale.abs()
    
    # 权重的通道最大值 (沿输出维度取最大)
    weight_max = weight.abs().max(dim=0)[0]
    
    # 计算平滑因子
    # s = act_max^α / weight_max^(1-α)
    smooth_factor = (act_max.pow(alpha) / (weight_max.pow(1 - alpha) + 1e-5)).clamp(min=1e-5)
    
    # 平滑权重: W' = W × diag(s)
    smoothed_weight = weight * smooth_factor.unsqueeze(0)
    
    return smoothed_weight, smooth_factor


# ============================================================
# SmoothQuant 演示
# ============================================================
print("=" * 60)
print("SmoothQuant 演示")
print("=" * 60)

# 模拟有异常值的激活
weight = torch.randn(128, 64)
activation_scale = torch.randn(64).abs()
activation_scale[10] = 100.0  # 模拟异常值 (outlier)
activation_scale[20] = 80.0   # 另一个异常值

print("\n原始数据:")
print(f"  权重范围: [{weight.min():.3f}, {weight.max():.3f}]")
print(f"  激活值最大值范围: [{activation_scale.min():.3f}, {activation_scale.max():.3f}]")
print(f"  激活值异常值: 通道10={activation_scale[10]:.1f}, 通道20={activation_scale[20]:.1f}")

# 应用 SmoothQuant
smoothed_weight, smooth_factor = smooth_quant(weight, activation_scale, alpha=0.5)

print("\nSmoothQuant 后:")
print(f"  平滑后权重范围: [{smoothed_weight.min():.3f}, {smoothed_weight.max():.3f}]")
print(f"  平滑因子范围: [{smooth_factor.min():.3f}, {smooth_factor.max():.3f}]")

# 验证: 平滑后激活值范围
smoothed_activation = activation_scale / smooth_factor
print(f"  平滑后激活值范围: [{smoothed_activation.min():.3f}, {smoothed_activation.max():.3f}]")

print("\n✓ 激活值异常值被平滑，更容易量化!")

<cell_type>markdown</cell_type>## 3. Movement Pruning

**核心概念**: 基于训练过程中权重的变化方向进行剪枝，而非仅看权重大小

```
┌─────────────────────────────────────────────────────────────┐
│                   Movement Pruning 原理                      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  传统幅度剪枝的问题:                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  只看权重大小: importance = |w|                     │   │
│  │  问题: 小权重可能正在变大 (重要)                    │   │
│  │        大权重可能正在变小 (不重要)                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  Movement Pruning 的改进:                                   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  看权重变化方向: movement = w × ∇w                  │   │
│  │                                                     │   │
│  │  movement > 0: 权重在远离零 → 重要，保留           │   │
│  │  movement < 0: 权重在靠近零 → 不重要，可剪枝       │   │
│  │                                                     │   │
│  │  累积多个训练步骤的 movement 分数                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  优点: 更准确地识别重要权重                                 │
│  缺点: 需要在训练过程中计算                                │
│  适用: 微调阶段的剪枝                                      │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Movement Pruning 实现
# ============================================================

class MovementPruner:
    """
    Movement Pruning 剪枝器
    
    基于训练过程中权重的变化方向进行剪枝
    
    工作原理:
    ┌─────────────────────────────────────────────────────────┐
    │  1. 在每个训练步骤后，计算 movement = w × ∇w          │
    │  2. 累积 movement 分数                                 │
    │  3. 根据累积分数决定剪枝                               │
    │     - 分数高 (远离零): 保留                           │
    │     - 分数低 (靠近零): 剪枝                           │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, model):
        """
        参数:
            model: 待剪枝的模型
        """
        self.model = model
        self.scores = {}
        
        # 为每个权重参数初始化分数
        for name, param in model.named_parameters():
            if 'weight' in name:
                self.scores[name] = torch.zeros_like(param)
    
    def update_scores(self):
        """
        更新 movement 分数
        
        在每个训练步骤的 backward() 之后调用
        """
        for name, param in self.model.named_parameters():
            if name in self.scores and param.grad is not None:
                # movement = w × ∇w
                # 正值: 权重在远离零 (重要)
                # 负值: 权重在靠近零 (可剪枝)
                movement = param.data * param.grad
                self.scores[name] += movement
    
    def get_mask(self, sparsity):
        """
        根据累积分数生成剪枝掩码
        
        参数:
            sparsity: 剪枝比例 (0-1)
            
        返回:
            dict: {参数名: 掩码}
        """
        masks = {}
        for name, score in self.scores.items():
            # 分数低于阈值的被剪枝
            threshold = torch.quantile(score.flatten(), sparsity)
            masks[name] = (score > threshold).float()
        return masks


# ============================================================
# Movement Pruning 演示
# ============================================================
print("=" * 60)
print("Movement Pruning 演示")
print("=" * 60)

# 创建模型和剪枝器
model = SimpleNet()
pruner = MovementPruner(model)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print("\n模拟训练过程，累积 movement 分数...")

# 模拟训练
for step in range(10):
    x = torch.randn(32, 64)
    y = torch.randint(0, 10, (32,))
    
    optimizer.zero_grad()
    output = model(x)
    loss = F.cross_entropy(output, y)
    loss.backward()
    
    # 更新 movement 分数
    pruner.update_scores()
    optimizer.step()

# 生成剪枝掩码
masks = pruner.get_mask(sparsity=0.3)

print("\n剪枝结果 (30% 稀疏度):")
print("-" * 40)
for name, mask in masks.items():
    sparsity = 1 - mask.mean().item()
    print(f"  {name}: 稀疏度 = {sparsity:.2%}")

<cell_type>markdown</cell_type>## 4. 自蒸馏 (Self-Distillation)

**核心概念**: 模型自己作为教师，深层指导浅层，无需额外的教师模型

```
┌─────────────────────────────────────────────────────────────┐
│                   自蒸馏原理                                 │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  传统蒸馏的问题:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  需要额外的大型教师模型                             │   │
│  │  教师模型训练成本高                                 │   │
│  │  教师-学生架构需要匹配                              │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  自蒸馏解决方案:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  模型自己作为教师:                                  │   │
│  │                                                     │   │
│  │  输入 → Layer1 → Layer2 → Layer3 → 最终输出        │   │
│  │           ↓        ↓                                │   │
│  │         Exit1    Exit2    (早期退出分类器)          │   │
│  │           ↓        ↓                                │   │
│  │         学习最终输出的软标签                        │   │
│  │                                                     │   │
│  │  深层知识 → 浅层                                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  优点: 无需额外教师模型，支持早期退出推理                   │
│  缺点: 需要修改模型架构                                    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 自蒸馏实现
# ============================================================

class SelfDistillationNet(nn.Module):
    """
    自蒸馏网络
    
    在中间层添加早期退出分类器，深层指导浅层
    
    架构:
    ┌─────────────────────────────────────────────────────────┐
    │  输入 → Layer1 → Layer2 → Layer3 → 最终分类头          │
    │           ↓        ↓                                    │
    │         Exit1    Exit2    (早期退出分类器)              │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, input_dim=64, hidden_dim=128, num_classes=10):
        super().__init__()
        # 主干网络
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, num_classes)  # 最终分类头
        
        # 早期退出分类器 (用于自蒸馏)
        self.exit1 = nn.Linear(hidden_dim, num_classes)  # Layer1 后的退出
        self.exit2 = nn.Linear(hidden_dim, num_classes)  # Layer2 后的退出
    
    def forward(self, x, return_exits=False):
        """
        前向传播
        
        参数:
            x: 输入
            return_exits: 是否返回早期退出的输出
            
        返回:
            如果 return_exits=False: 只返回最终输出
            如果 return_exits=True: 返回 (最终输出, [早期退出输出列表])
        """
        h1 = F.relu(self.layer1(x))
        h2 = F.relu(self.layer2(h1))
        h3 = F.relu(self.layer3(h2))
        final = self.head(h3)
        
        if return_exits:
            exit1 = self.exit1(h1)
            exit2 = self.exit2(h2)
            return final, [exit1, exit2]
        return final


def self_distillation_loss(final_output, exit_outputs, labels, temperature=3.0, alpha=0.5):
    """
    自蒸馏损失函数
    
    损失 = 硬标签损失 + α × 蒸馏损失
    
    参数:
        final_output: 最终层输出 (教师)
        exit_outputs: 早期退出输出列表 (学生)
        labels: 真实标签
        temperature: 温度参数
        alpha: 蒸馏损失权重
        
    返回:
        总损失
    """
    # 硬标签损失 (最终输出)
    hard_loss = F.cross_entropy(final_output, labels)
    
    # 软标签 (来自最终输出，作为教师)
    soft_teacher = F.softmax(final_output.detach() / temperature, dim=-1)
    
    # 蒸馏损失 (早期退出学习最终输出)
    distill_loss = 0
    for exit_out in exit_outputs:
        soft_student = F.log_softmax(exit_out / temperature, dim=-1)
        distill_loss += F.kl_div(soft_student, soft_teacher, reduction='batchmean')
    
    # 总损失
    total_loss = hard_loss + alpha * distill_loss * (temperature ** 2)
    
    return total_loss


# ============================================================
# 自蒸馏训练演示
# ============================================================
print("=" * 60)
print("自蒸馏训练演示")
print("=" * 60)

model = SelfDistillationNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("\n训练过程:")
print("-" * 40)

for epoch in range(5):
    x = torch.randn(32, 64)
    y = torch.randint(0, 10, (32,))
    
    # 获取最终输出和早期退出输出
    final, exits = model(x, return_exits=True)
    
    # 计算自蒸馏损失
    loss = self_distillation_loss(final, exits, y, temperature=3.0, alpha=0.5)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"  Epoch {epoch+1}: Loss = {loss.item():.4f}")

print("\n✓ 自蒸馏训练完成!")
print("  早期退出分类器学习了深层的知识")

<cell_type>markdown</cell_type>## 5. KV Cache 优化

**核心概念**: 缓存 Transformer 自回归推理中的 Key-Value，避免重复计算

```
┌─────────────────────────────────────────────────────────────┐
│                   KV Cache 原理                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  问题: 自回归生成每个 token 都需要重新计算所有 KV          │
│                                                             │
│  无 KV Cache:                                               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  生成 token 1: 计算 K1, V1                          │   │
│  │  生成 token 2: 重新计算 K1, V1, K2, V2              │   │
│  │  生成 token 3: 重新计算 K1, V1, K2, V2, K3, V3      │   │
│  │  ...                                                │   │
│  │  计算量: O(n²) 随序列长度平方增长!                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  有 KV Cache:                                               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  生成 token 1: 计算 K1, V1 → 缓存                   │   │
│  │  生成 token 2: 只计算 K2, V2 → 追加到缓存           │   │
│  │  生成 token 3: 只计算 K3, V3 → 追加到缓存           │   │
│  │  ...                                                │   │
│  │  计算量: O(n) 线性增长!                             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  KV Cache 大小计算:                                         │
│  size = 2 × num_layers × seq_len × hidden_dim × dtype_size │
│                                                             │
│  例: LLaMA-7B, seq_len=2048, FP16                          │
│  size = 2 × 32 × 2048 × 4096 × 2 bytes ≈ 1 GB             │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# KV Cache 实现
# ============================================================

class KVCache:
    """
    Key-Value 缓存
    
    用于 Transformer 自回归推理，避免重复计算
    
    工作原理:
    ┌─────────────────────────────────────────────────────────┐
    │  1. 初始化空缓存                                       │
    │  2. 每生成一个 token，将新的 K, V 追加到缓存           │
    │  3. 注意力计算时使用完整的缓存 K, V                    │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, batch_size, max_length, num_heads, head_dim):
        """
        参数:
            batch_size: 批次大小
            max_length: 最大序列长度
            num_heads: 注意力头数
            head_dim: 每个头的维度
        """
        # 预分配缓存空间
        self.k_cache = torch.zeros(batch_size, num_heads, max_length, head_dim)
        self.v_cache = torch.zeros(batch_size, num_heads, max_length, head_dim)
        self.current_length = 0
    
    def update(self, new_k, new_v):
        """
        更新缓存
        
        参数:
            new_k: 新的 Key [batch, heads, seq_len, head_dim]
            new_v: 新的 Value [batch, heads, seq_len, head_dim]
        """
        seq_len = new_k.size(2)
        self.k_cache[:, :, self.current_length:self.current_length+seq_len] = new_k
        self.v_cache[:, :, self.current_length:self.current_length+seq_len] = new_v
        self.current_length += seq_len
    
    def get(self):
        """
        获取当前缓存的 K, V
        
        返回:
            (k_cache, v_cache): 截止到当前长度的缓存
        """
        return (
            self.k_cache[:, :, :self.current_length],
            self.v_cache[:, :, :self.current_length]
        )


class CachedAttention(nn.Module):
    """
    带 KV Cache 的注意力层
    """
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Q, K, V 投影
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, x, kv_cache=None):
        """
        前向传播
        
        参数:
            x: 输入 [batch, seq_len, d_model]
            kv_cache: KV 缓存 (可选)
            
        返回:
            输出 [batch, seq_len, d_model]
        """
        B, T, C = x.shape
        
        # 计算 Q, K, V
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 使用 KV Cache
        if kv_cache is not None:
            kv_cache.update(k, v)
            k, v = kv_cache.get()
        
        # 注意力计算
        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        
        return self.out_proj(out)


# ============================================================
# KV Cache 演示
# ============================================================
print("=" * 60)
print("KV Cache 演示")
print("=" * 60)

attn = CachedAttention(d_model=64, num_heads=4)
cache = KVCache(batch_size=1, max_length=100, num_heads=4, head_dim=16)

print("\n模拟自回归生成 (每次一个 token):")
print("-" * 40)

# 模拟自回归生成
for i in range(5):
    x = torch.randn(1, 1, 64)  # 每次只输入一个 token
    out = attn(x, kv_cache=cache)
    print(f"  Step {i+1}: 输入 1 token, Cache 长度 = {cache.current_length}")

print("\n✓ KV Cache 避免了重复计算历史 token 的 K, V!")

<cell_type>markdown</cell_type>## 6. Speculative Decoding (投机解码)

**核心概念**: 使用小模型快速生成候选 token，大模型并行验证，加速自回归生成

```
┌─────────────────────────────────────────────────────────────┐
│                   Speculative Decoding 原理                  │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  传统自回归生成的问题:                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  大模型生成 token 1 → 大模型生成 token 2 → ...     │   │
│  │  每个 token 都需要完整的大模型前向传播              │   │
│  │  速度慢！                                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  Speculative Decoding 解决方案:                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 小模型 (Draft) 快速生成 k 个候选 token          │   │
│  │  2. 大模型 (Target) 并行验证这 k 个 token           │   │
│  │  3. 接受正确的，拒绝错误的                          │   │
│  │  4. 从第一个错误位置重新开始                        │   │
│  │                                                     │   │
│  │  加速原理:                                          │   │
│  │  - 小模型生成快 (参数少)                           │   │
│  │  - 大模型验证可以并行 (一次前向验证 k 个)          │   │
│  │  - 如果小模型准确率高，大部分 token 被接受         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  加速比: 通常 2-3x，取决于小模型准确率                     │
│                                                             │
│  接受条件 (保证输出分布不变):                               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  如果 P_target(token) >= P_draft(token):            │   │
│  │      接受该 token                                   │   │
│  │  否则:                                              │   │
│  │      以概率 P_target/P_draft 接受                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Speculative Decoding 实现
# ============================================================

class SimpleLM(nn.Module):
    """
    简单语言模型
    
    用于演示 Speculative Decoding
    """
    
    def __init__(self, vocab_size, d_model, n_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        h = self.embed(x)
        for layer in self.layers:
            h = F.relu(layer(h))
        return self.head(h)


class SpeculativeDecoder:
    """
    Speculative Decoding 解码器
    
    使用小模型加速大模型推理
    
    工作流程:
    ┌─────────────────────────────────────────────────────────┐
    │  1. Draft 模型快速生成 k 个候选 token                  │
    │  2. Target 模型并行验证这 k 个 token                   │
    │  3. 比较概率，决定接受/拒绝                            │
    │  4. 返回接受的 token                                   │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, draft_model, target_model, k=4):
        """
        参数:
            draft_model: 小模型 (快速但不太准确)
            target_model: 大模型 (慢但准确)
            k: 每次投机生成的 token 数
        """
        self.draft = draft_model
        self.target = target_model
        self.k = k
    
    @torch.no_grad()
    def generate_step(self, input_ids):
        """
        执行一步 Speculative Decoding
        
        参数:
            input_ids: 输入 token 序列
            
        返回:
            (output_ids, accepted): 输出序列和接受的 token 数
        """
        # ============================================================
        # 步骤 1: Draft 模型快速生成 k 个候选 token
        # ============================================================
        draft_tokens = []
        draft_probs = []
        current = input_ids.clone()
        
        for _ in range(self.k):
            logits = self.draft(current)[:, -1]
            probs = F.softmax(logits, dim=-1)
            token = torch.argmax(probs, dim=-1, keepdim=True)
            draft_tokens.append(token)
            draft_probs.append(probs.gather(-1, token))
            current = torch.cat([current, token], dim=1)
        
        # ============================================================
        # 步骤 2: Target 模型并行验证
        # ============================================================
        target_logits = self.target(current)
        target_probs = F.softmax(target_logits[:, -self.k-1:-1], dim=-1)
        
        # ============================================================
        # 步骤 3: 接受/拒绝判断
        # ============================================================
        accepted = 0
        for i, (token, draft_p) in enumerate(zip(draft_tokens, draft_probs)):
            target_p = target_probs[:, i].gather(-1, token)
            # 如果 target 概率 >= draft 概率，接受
            if (target_p >= draft_p).all():
                accepted += 1
            else:
                break
        
        # 返回接受的 token
        return current[:, :input_ids.size(1) + accepted + 1], accepted


# ============================================================
# Speculative Decoding 演示
# ============================================================
print("=" * 60)
print("Speculative Decoding 演示")
print("=" * 60)

vocab_size = 100

# 创建 Draft 模型 (小) 和 Target 模型 (大)
draft = SimpleLM(vocab_size, d_model=32, n_layers=2)   # 小模型
target = SimpleLM(vocab_size, d_model=64, n_layers=4)  # 大模型

print(f"\nDraft 模型参数量: {sum(p.numel() for p in draft.parameters()):,}")
print(f"Target 模型参数量: {sum(p.numel() for p in target.parameters()):,}")

# 创建解码器
decoder = SpeculativeDecoder(draft, target, k=4)

# 测试
input_ids = torch.randint(0, vocab_size, (1, 5))

print(f"\n输入序列长度: {input_ids.size(1)}")

output, accepted = decoder.generate_step(input_ids)

print(f"输出序列长度: {output.size(1)}")
print(f"接受的 token 数: {accepted}")
print(f"投机成功率: {accepted}/{decoder.k} = {accepted/decoder.k:.0%}")

print("\n✓ Speculative Decoding 可以显著加速 LLM 推理!")

<cell_type>markdown</cell_type>## 总结

本教程介绍了六种高级模型优化技术：

### 技术对比

| 技术 | 适用场景 | 核心思想 | 优势 |
|:-----|:---------|:---------|:-----|
| 混合精度量化 | 精度敏感模型 | 敏感层高精度，其他层低精度 | 平衡精度和效率 |
| SmoothQuant | LLM 量化 | 平滑激活值异常 | 解决 outlier 问题 |
| Movement Pruning | 微调剪枝 | 基于权重变化方向 | 保留重要权重 |
| 自蒸馏 | 模型压缩 | 深层指导浅层 | 无需额外教师 |
| KV Cache | Transformer 推理 | 缓存历史 K, V | 避免重复计算 |
| Speculative Decoding | LLM 推理 | 小模型投机，大模型验证 | 加速自回归生成 |

### 选择建议

```
┌─────────────────────────────────────────────────────────────┐
│                   技术选择指南                               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  场景: LLM 量化部署                                         │
│  └── 推荐: SmoothQuant + 混合精度量化                      │
│                                                             │
│  场景: 微调后剪枝                                           │
│  └── 推荐: Movement Pruning                                │
│                                                             │
│  场景: 无教师模型的压缩                                     │
│  └── 推荐: 自蒸馏                                          │
│                                                             │
│  场景: LLM 推理加速                                         │
│  └── 推荐: KV Cache + Speculative Decoding                 │
│                                                             │
│  场景: 边缘设备部署                                         │
│  └── 推荐: 混合精度量化 + 结构化剪枝                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 最佳实践

```
高级优化检查清单:
✓ 分析模型各层敏感度，制定混合精度策略
✓ 对 LLM 使用 SmoothQuant 处理激活值异常
✓ 微调时使用 Movement Pruning 而非幅度剪枝
✓ 考虑自蒸馏减少对教师模型的依赖
✓ Transformer 推理必须使用 KV Cache
✓ 长文本生成考虑 Speculative Decoding

常见陷阱:
✗ 对所有层使用相同精度
✗ 忽略 LLM 激活值的异常值问题
✗ 只看权重大小进行剪枝
✗ 自回归生成不使用 KV Cache
```

### 参考论文

- **SmoothQuant**: [SmoothQuant: Accurate and Efficient Post-Training Quantization for LLMs](https://arxiv.org/abs/2211.10438)
- **Movement Pruning**: [Movement Pruning: Adaptive Sparsity by Fine-Tuning](https://arxiv.org/abs/2005.07683)
- **Speculative Decoding**: [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)